# **Binary Classification1**

### 1) 실습 개요
이번 실습에서는 학습한 선형 회귀 모델을 새로운 데이터에 대해 평가하는 방법을 공부합니다.

### 2) 실습 진행 목적 및 배경
머신러닝에 있어서 학습만큼 중요한 것은 공정한 평가입니다. 따라서 본 실습에서는 앞선 강의에서 배운 방법으로 구축한 선형 회귀 모델을 올바르게 평가하는 방법을 공부합니다.

### 3) 실습 수행으로 얻어갈 수 있는 역량
- 테스트 코드를 작성하여 새로운 데이터에 대한 평가를 진행할 수 있다.

### 4) 실습 핵심 내용
&nbsp;&nbsp; 1.1 테스트<br>

### 5) 데이터셋 개요 및 저작권 정보

- 사용 데이터셋: [Salary Dataset](https://www.kaggle.com/datasets/abhishek14398/salary-dataset-simple-linear-regression)
  - 연차(YearsExperience)에 따른 임금(Salary) 데이터셋입니다.
- 저작권 정보: [CC0 1.0 Universal](https://creativecommons.org/publicdomain/zero/1.0/)

### 6) Required Package
```python
torch >= 2.3.0
pandas >= 2.0.3
scikit-learn >= 1.2.2
numpy >= 1.25.2
```

# **1. 선형 회귀 모델의 테스트**

In [ ]:
import torch

In [ ]:
# Donwload dataset from kaggle
!kaggle datasets download -d abhishek14398/salary-dataset-simple-linear-regression
# unzip zip file
!unzip salary-dataset-simple-linear-regression.zip

Dataset URL: https://www.kaggle.com/datasets/abhishek14398/salary-dataset-simple-linear-regression
License(s): CC0-1.0
  0% 0.00/457 [00:00<?, ?B/s]
100% 457/457 [00:00<00:00, 964kB/s]
Archive:  salary-dataset-simple-linear-regression.zip
  inflating: Salary_dataset.csv      


In [ ]:
# 트레이닝 데이터의 코드 표현

import pandas as pd
data = pd.read_csv("Salary_dataset.csv", sep = ",", header = 0) # 데이터 불러오기

# 특징 변수와 목적 변수로 분리

x = data.iloc[:, 1].values # 두 번째 열을 특징 변수로 사용 (YearsExperience)
t = data.iloc[:, 2].values # 세 번째 열을 목적 변수로 사용 (Salary)

# 데이터 표준화의 코드 표현 실습

from sklearn.preprocessing import StandardScaler

scaler_x = StandardScaler()
x_scaled = scaler_x.fit_transform(x.reshape(-1, 1))

scaler_t = StandardScaler()
t_scaled = scaler_t.fit_transform(t.reshape(-1, 1))

# 데이터를 표준화한 numpy 배열을 Tensor로 변환

x_tensor = torch.tensor(x_scaled, dtype=torch.float32).view(-1, 1)  # 2차원 형태로 변환
t_tensor = torch.tensor(t_scaled, dtype=torch.float32).view(-1, 1)

In [ ]:
# 선형 회귀 모델 코드 표현 실습

import torch.nn as nn

class LinearRegressionModel(nn.Module):
    def __init__(self):
        super(LinearRegressionModel, self).__init__()
        self.linear = nn.Linear(1, 1)  # 입력과 출력이 모두 1개인 선형 회귀 모델

    def forward(self, x):
        y = self.linear(x) # 입력 데이터를 선형 계층을 통해 예측값 계산
        return y

# 모델 초기화

model = LinearRegressionModel()

# 확률적 경사하강법 코드 표현 실습
# 손실 함수 및 옵티마이저 정의

# GPU 지원

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
x_tensor = x_tensor.to(device)
t_tensor = t_tensor.to(device)

import torch.optim as optim

loss_function = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr = 0.01)  # 학습률을 적절히 설정

# 에포크 코드 표현 실습
# 모델 반복 학습

num_epochs = 1000  # 에포크 수를 증가
loss_list = []  # 손실 값을 저장할 리스트

for epoch in range(num_epochs):
    y = model(x_tensor)  # 예측 변수 계산
    loss = loss_function(y, t_tensor)  # 손실 값 계산

    # 확률적 경사하강법 작동원리의 코드 표현 실습

    optimizer.zero_grad()  # 이전 단계에서 계산된 경사(기울기)를 0으로 초기화
    loss.backward()  # 현재 손실(loss)에 대한 경사(기울기)를 계산 (역전파 수행)
    optimizer.step()  # 계산된 경사(기울기)를 사용하여 가중치를 업데이트

    # 손실 값을 저장

    loss_list.append(loss.item())

    if (epoch+1) % 100 == 0:
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item()}')

        # 디버깅 정보 출력

        for name, param in model.named_parameters():
            print(f'{name}: {param.data}')

# 가중치와 바이어스를 기존 스케일로 변환하는 코드 표현

weight = model.linear.weight.item()
bias = model.linear.bias.item()

original_weight = weight * (scaler_t.scale_[0] / scaler_x.scale_[0])
original_bias = bias * scaler_t.scale_[0] + scaler_t.mean_[0] - original_weight * scaler_x.mean_[0]

print(f'기존 스케일의 가중치: {original_weight}')
print(f'기존 스케일의 바이어스: {original_bias}')

Epoch [100/1000], Loss: 0.10123948752880096
linear.weight: tensor([[0.7459]])
linear.bias: tensor([0.0435])
Epoch [200/1000], Loss: 0.044066883623600006
linear.weight: tensor([[0.9474]])
linear.bias: tensor([0.0058])
Epoch [300/1000], Loss: 0.043061330914497375
linear.weight: tensor([[0.9742]])
linear.bias: tensor([0.0008])
Epoch [400/1000], Loss: 0.043043654412031174
linear.weight: tensor([[0.9777]])
linear.bias: tensor([0.0001])
Epoch [500/1000], Loss: 0.0430433414876461
linear.weight: tensor([[0.9782]])
linear.bias: tensor([1.3451e-05])
Epoch [600/1000], Loss: 0.043043337762355804
linear.weight: tensor([[0.9782]])
linear.bias: tensor([1.7814e-06])
Epoch [700/1000], Loss: 0.043043334037065506
linear.weight: tensor([[0.9782]])
linear.bias: tensor([2.3412e-07])
Epoch [800/1000], Loss: 0.043043334037065506
linear.weight: tensor([[0.9782]])
linear.bias: tensor([3.8077e-08])
Epoch [900/1000], Loss: 0.043043334037065506
linear.weight: tensor([[0.9782]])
linear.bias: tensor([1.9655e-08])
Ep

**1.1 테스트**

In [ ]:
# 테스트 데이터의 코드 표현 실습

import numpy as np

def predict_test_data(test_data):
    # 테스트 데이터 표준화
    test_scaled = scaler_x.transform(test_data.reshape(-1, 1))
    test_tensor = torch.tensor(test_scaled, dtype=torch.float32).view(-1, 1).to(device)

    # 모델을 사용하여 예측
    model.eval()  # 평가 모드로 전환
    with torch.no_grad():
        predictions_scaled = model(test_tensor)

    # 표준화 해제
    predictions = scaler_t.inverse_transform(predictions_scaled.cpu().numpy())
    return predictions

# 예측할 테스트 데이터
test_years_experience = np.array([1.0, 2.0, 7.0])  # 1년, 2년, 7년의 경력을 가진  데이터를 사용
predicted_salaries = predict_test_data(test_years_experience)

# 결과 출력
for YearsExperience, salary in zip(test_years_experience, predicted_salaries):
    print(f'YearsExperience: {YearsExperience}, Predicted Salary: {salary[0]:.0f}')

YearsExperience: 1.0, Predicted Salary: 34298
YearsExperience: 2.0, Predicted Salary: 43748
YearsExperience: 7.0, Predicted Salary: 90998


## 콘텐츠 라이선스

<hr style="height:5px;border:none;color:#5F71F7;background-color:#5F71F7">

<font color='red'><b>WARNING</font> : 본 교육 콘텐츠의 지식재산권은 재단법인 네이버커넥트에 귀속됩니다. 본 콘텐츠를 어떠한 경로로든 외부로 유출 및 수정하는 행위를 엄격히 금합니다. 다만, 비영리적 교육 및 연구활동에 한정되어 사용할 수 있으나 재단의 허락을 받아야 합니다. 이를 위반하는 경우, 관련 법률에 따라 책임을 질 수 있습니다. </b>